# Best models · Fraud history forest and activity sports ensemble
Run top to bottom to evaluate and export the fraud forest and the sports-aware activity classifier. Each task reports two metrics. The last cell reloads both fitted files and shows the prediction API.

**Inputs.** Put `cct_train.zip` (containing `cct_train.csv`) and the headerless `har_train.csv` beside this notebook or leave them in the project's `Docs and Data` folder. Colab prompts for missing files. Set `DATA_DIR` to use another location. Outputs go to the `best_models` folder in the working directory.

**Environment.** The saved pickles were built with Python 3.12. For the matching package versions, uncomment and run the install cell in a fresh local Jupyter or Colab runtime, then restart the kernel. To predict with the supplied files, run the imports and follow the loading example at the end. Only load trusted pickle files.

In [1]:
# Optional setup for a fresh environment. Use the kernel's pip so packages land in the right environment.
# %pip install numpy==2.5.5 pandas==3.0.1 scipy==1.18.1 scikit-learn==1.9.1 imbalanced-learn==0.14.2 xgboost==3.4.1 lightgbm==4.7.0 cloudpickle==3.1.3 threadpoolctl==3.8.0

In [2]:
import os
import pickle
import sys
import zipfile
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

SEED = 610  # Forest, grouped splits, and SMOTE. Native booster seeds are retained below.
WORKERS = min(8, os.cpu_count() or 1)
DATA_DIR = Path.cwd()
OUTPUT_DIR = Path.cwd() / "best_models"
OUTPUT_DIR.mkdir(exist_ok=True)

def input_path(name):
    for root in (DATA_DIR, DATA_DIR.parent):
        if (root / name).is_file():
            return root / name
        match = next((root / "Docs and Data").rglob(name), None)
        if match:
            return match
    if "google.colab" in sys.modules:
        from google.colab import files
        print("Upload", name)
        files.upload()
        if Path(name).is_file():
            return Path(name)
    raise FileNotFoundError(f"Place {name} in {DATA_DIR} or set DATA_DIR.")

## 1 · Credit card fraud
The random forest compares each payment with past card, account, merchant and category activity. The two category-share features compare the last seven days with the preceding 28. History uses only earlier timestamps and never uses fraud labels. Three expanding development windows set the positive-class F1 threshold. Transactions from September 2022 onward are scored once with that threshold.

In [3]:
entity_fields = ["cc_num", "acct_num", "first", "last", "dob", "merchant", "category"]
event_fields = entity_fields + ["unix_time", "trans_time", "trans_date", "amt", "lat", "long", "merch_lat", "merch_long"]
with zipfile.ZipFile(input_path("cct_train.zip")) as archive:
    member, = [n for n in archive.namelist() if Path(n).name == "cct_train.csv"]
    with archive.open(member) as stream:
        fraud = pd.read_csv(stream, usecols=event_fields + ["is_fraud"], dtype={k:"string" for k in entity_fields})
fraud = fraud.sort_values("unix_time", kind="stable").reset_index(drop=True)

In [4]:
def prior_counts(times, days):
    """Count earlier events in a trailing window, excluding simultaneous events."""
    return times.searchsorted(times) - times.searchsorted(times - days * 86400)

def fraud_features(events):
    # Return features in input order, including when called on a shuffled batch.
    d = events.reset_index(drop=True).sort_values("unix_time", kind="stable")
    x = pd.DataFrame(index=d.index)
    x["amount"] = d.amt
    x["hour"] = d.trans_time.str.slice(0, 2).astype(int)
    lat1, lat2 = np.radians(d.lat), np.radians(d.merch_lat)
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(np.radians(d.merch_long-d.long)/2)**2
    x["home_merchant_km"] = 12742 * np.arcsin(np.sqrt(a.clip(0, 1)))

    d["time_band"] = x["hour"] // 6
    entities = {"card":["cc_num"], "card_category":["cc_num","category"],
                "account":["acct_num"], "person":["first","last","dob"],
                "card_merchant":["cc_num","merchant"], "category":["category"], "merchant":["merchant"],
                "merchant_time":["merchant","time_band"], "card_time":["cc_num","time_band"]}
    for prefix, keys in entities.items():
        # Collapse simultaneous events so their within-second ordering is immaterial.
        group_keys = keys + ["unix_time"]
        present = d[keys].notna().all(axis=1)
        batch = (d.loc[present].groupby(group_keys, sort=False, observed=True)
                 .amt.agg(total="sum", n="size").reset_index())
        groups = batch.groupby(keys, sort=False, observed=True)
        count = groups.n.cumsum() - batch.n
        total = groups.total.cumsum() - batch.total
        batch["count"] = count
        batch["mean"] = total / count.replace(0, np.nan)
        batch["elapsed"] = groups.unix_time.diff()
        prior = d[group_keys].merge(batch[group_keys + ["count","mean","elapsed"]],
                                    on=group_keys, how="left", sort=False, validate="many_to_one")
        prior.index = d.index
        x[prefix+"_count"] = prior["count"].fillna(0)
        x[prefix+"_amount_ratio"] = d.amt / prior["mean"].replace(0, np.nan)
        if prefix == "card":
            x["card_elapsed"] = prior["elapsed"]
    # Prefix sums and binary search make strict trailing windows inexpensive per card.
    for name in ("hour_count","day_count","day_spend"):
        x[name] = 0.
    x["reference_days_observed"] = 0.
    x["day_vs_prior28_ratio"] = np.nan
    for _, block in d.groupby("cc_num", sort=False):
        times = block.unix_time.to_numpy()
        end = times.searchsorted(times, side="left")
        hour = times.searchsorted(times-3600)
        day = times.searchsorted(times-86400)
        sums = np.r_[0., block.amt.to_numpy().cumsum()]
        x.loc[block.index, ["hour_count","day_count","day_spend"]] = np.column_stack(
            [end-hour, end-day, sums[end]-sums[day]])
        ref_start = times.searchsorted(times-29*86400, side="left")
        ref_total = sums[day]-sums[ref_start]
        coverage = np.clip((times-times[0])/86400-1, 0, 28)
        ratio = np.full(len(block), np.nan)
        np.divide(sums[end]-sums[day], ref_total/28, out=ratio,
                  where=(coverage == 28) & (ref_total > 0))
        x.loc[block.index, ["reference_days_observed","day_vs_prior28_ratio"]] = np.column_stack([coverage,ratio])
    # Count category and overall traffic over aligned historical windows.
    t = d.unix_time.to_numpy()
    c7=np.zeros(len(t),int); c35=c7.copy()
    for idx in d.reset_index(drop=True).groupby("category",sort=False).indices.values():
        c7[idx]=prior_counts(t[idx],7); c35[idx]=prior_counts(t[idx],35)
    n7,n35=prior_counts(t,7),prior_counts(t,35)
    x["recent_category_share"]=(c7+.5)/(n7+7)
    x["category_share_change"]=np.log(x.recent_category_share.to_numpy()/((c35-c7+.5)/(n35-n7+7)))
    return x.sort_index().astype("float32")

In [5]:
basic = ["amount","hour"]
card = basic + ["card_amount_ratio","card_category_amount_ratio","hour_count","day_count","card_elapsed"]
extended = card + ["day_spend","card_count","card_merchant_count","home_merchant_km",
                   "account_count","account_amount_ratio","person_count","person_amount_ratio",
                   "category_amount_ratio"]

merchant = ["merchant_count", "merchant_amount_ratio"]
trend = ["reference_days_observed", "day_vs_prior28_ratio"]
fraud_columns = extended + merchant + trend + [
    "card_merchant_amount_ratio", "card_category_count", "merchant_time_count",
    "merchant_time_amount_ratio", "card_time_amount_ratio", "recent_category_share", "category_share_change",
]

def new_forest():
    """Learn missing-value medians and flags from the training period only."""
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                         RandomForestClassifier(n_estimators=100, max_samples=0.5,
                                                min_samples_leaf=2, n_jobs=WORKERS, random_state=SEED))

def select_threshold(y, scores):
    """Maximize positive-class F1 over distinct scores; break ties by accuracy."""
    order = np.argsort(-scores, kind="stable")
    ranked = scores[order]
    ends = np.flatnonzero(np.r_[ranked[:-1] != ranked[1:], True])
    tp = np.r_[0, y[order].cumsum()[ends]]
    alerts = np.r_[0, ends + 1]
    f1 = 2 * tp / np.maximum(1, alerts + y.sum())
    tied = np.flatnonzero(f1 == f1.max())
    winner = tied[np.argmax((2 * tp - alerts)[tied])]
    return np.r_[np.inf, ranked[ends]][winner]

X = fraud_features(fraud[event_fields])[fraud_columns].to_numpy(dtype="float32")
y = fraud.is_fraud.to_numpy(dtype=int)
dates = pd.to_datetime(fraud.trans_date)
edges = pd.to_datetime(["2022-03-01", "2022-05-01", "2022-07-01", "2022-09-01"])
dev_y, dev_scores = [], []
for start, stop in zip(edges[:-1], edges[1:]):
    train, valid = dates < start, (dates >= start) & (dates < stop)
    forest = new_forest().fit(X[train], y[train])
    dev_scores.extend(forest.predict_proba(X[valid])[:, 1])
    dev_y.extend(y[valid])
threshold = select_threshold(np.asarray(dev_y), np.asarray(dev_scores))

later = dates >= edges[-1]
forest = new_forest().fit(X[~later], y[~later])
fraud_pred = forest.predict_proba(X[later])[:, 1] >= threshold
fraud_result = [accuracy_score(y[later], fraud_pred), f1_score(y[later], fraud_pred)]
print("Later-period accuracy and fraud F1:", fraud_result)

Later-period accuracy and fraud F1: [0.9998219855024993, 0.9749247743229689]


## 2 · Activity recognition
The multiclass booster sees standardized sensors and the distance-weighted class shares of the 15 nearest distinct training readings. A Tennis/Football specialist refines those two scores. A second booster trained on SMOTE–Tomek data contributes only to those same two scores. Identical sensor vectors stay in one fold. Each training row receives neighbor shares from a separate inner training fold to prevent self-matches.

In [6]:
def activity_booster(fold):
    """Recreate the previously selected native booster configuration for this fold."""
    if fold == 1:
        return LGBMClassifier(n_estimators=100, num_leaves=4, min_child_samples=16,
                              learning_rate=0.07832503615697724, max_bin=511,
                              colsample_bytree=0.8814460015050225, reg_alpha=0.0009765625,
                              reg_lambda=0.024037654653852953, n_jobs=WORKERS, verbose=-1)
    return XGBClassifier(n_estimators=56, max_leaves=4, max_depth=0, grow_policy="lossguide",
                         tree_method="hist", enable_categorical=True, verbosity=0, random_state=10242048,
                         min_child_weight=2.707170950755067, learning_rate=0.4810290992611119,
                         subsample=0.9659488304727971, colsample_bytree=0.9692046632582698,
                         reg_alpha=0.0009765625, reg_lambda=0.1384884162470523, n_jobs=WORKERS)

def neighbor_shares(ref, query, ids, groups, class_count):
    """One vote per distinct reading; conflicting labels split that reading's vote."""
    scale = StandardScaler().fit(ref)
    keep = scale.var_ > 0
    train, target = scale.transform(ref)[:, keep], scale.transform(query)[:, keep]
    unique, first, inverse = np.unique(groups, return_index=True, return_inverse=True)
    votes = np.zeros((len(unique), class_count))
    np.add.at(votes, (inverse, ids), 1)
    votes /= votes.sum(axis=1, keepdims=True)
    distance, near = NearestNeighbors(n_neighbors=15).fit(train[first]).kneighbors(target)
    weights = 1 / np.maximum(distance, 1e-8)
    return np.sum(votes[near] * weights[:, :, None], axis=1) / weights.sum(axis=1, keepdims=True)

def group_folds(x, y, groups, count, class_count):
    """Repeat the original deterministic split search to keep all training classes."""
    for seed in range(SEED, SEED + 200):
        folds = list(StratifiedGroupKFold(count, shuffle=True, random_state=seed).split(x, y, groups))
        if all(len(np.unique(y[a])) == class_count and
               (count == 3 or len(np.unique(y[b])) == class_count) for a, b in folds):
            return folds
    raise ValueError("No grouped split covered all required classes.")

In [7]:
def fit_activity_fold(x, y, groups, labels, fold):
    """Fit preprocessing, cross-fitted neighbor features and the three component boosters."""
    count = len(labels)
    pair = np.array([np.flatnonzero(labels == sport)[0] for sport in ("Football", "Tennis")])
    scale = StandardScaler().fit(x)
    sensors = scale.transform(x)
    meta = np.empty((len(x), count))
    for train, valid in group_folds(x, y, groups, 3, count):
        meta[valid] = neighbor_shares(x[train], x[valid], y[train], groups[train], count)
    share_scale = StandardScaler().fit(meta)
    augmented = np.column_stack((sensors, share_scale.transform(meta)))
    general = activity_booster(fold).fit(augmented, y, sample_weight=compute_sample_weight("balanced", y))
    sports = np.isin(y, pair)
    specialist = activity_booster(fold).fit(augmented[sports], (y[sports] == pair[1]).astype(int))

    # Lift classes below median support, then remove Tomek links within this training fold.
    counts = np.bincount(y)
    target = int(np.median(counts))
    strategy = {i: target for i, n in enumerate(counts) if n < target}
    sampler = SMOTETomek(smote=SMOTE(sampling_strategy=strategy, k_neighbors=min(3, counts.min()-1),
                                      random_state=SEED),
                         tomek=TomekLinks(sampling_strategy="all", n_jobs=1), random_state=SEED)
    xr, yr = sampler.fit_resample(sensors, y)
    resampled = activity_booster(fold).fit(xr, yr)
    return dict(x=x, y=y, groups=groups, count=count, pair=pair, scale=scale, share_scale=share_scale,
                general=general, specialist=specialist, resampled=resampled)

def activity_scores(model, x):
    """Return decision scores in training-label order. Only the two sports scores are blended."""
    sensors = model["scale"].transform(x)
    meta = neighbor_shares(model["x"], x, model["y"], model["groups"], model["count"])
    augmented = np.column_stack((sensors, model["share_scale"].transform(meta.copy())))
    scores = model["general"].predict_proba(augmented)
    pair = model["pair"]
    sports_mass = scores[:, pair].sum(axis=1, keepdims=True)
    refined = (scores[:, pair] + sports_mass * model["specialist"].predict_proba(augmented)) / 2
    scores[:, pair] = (refined + model["resampled"].predict_proba(sensors)[:, pair]) / 2
    return scores

In [8]:
har = pd.read_csv(input_path("har_train.csv"), header=None)
raw = har.iloc[:, :-1]
labels, targets = np.unique(har.iloc[:, -1], return_inverse=True)
groups = pd.util.hash_pandas_object(raw, index=False).to_numpy()
activity_folds = group_folds(raw, targets, groups, 4, len(labels))
oof = np.empty_like(targets)
fold_models = []
# Single-threaded BLAS reproduces the reference neighbor distances. Boosters retain their workers.
with threadpool_limits(limits=1, user_api="blas"):
    for fold, (train, valid) in enumerate(activity_folds):
        assert not np.intersect1d(groups[train], groups[valid]).size
        model = fit_activity_fold(raw.iloc[train].to_numpy(), targets[train], groups[train], labels, fold)
        oof[valid] = activity_scores(model, raw.iloc[valid].to_numpy()).argmax(axis=1)
        fold_models.append(model)
activity_result = [accuracy_score(targets, oof), f1_score(targets, oof, average="macro")]
print(pd.DataFrame([["Fraud · later period", *fraud_result, "fraud F1"],
                    ["Activity · grouped out-of-fold", *activity_result, "macro F1"]],
                   columns=["Evaluation", "Accuracy", "F1", "F1 definition"]).to_string(index=False))

                    Evaluation  Accuracy       F1 F1 definition
          Fraud · later period  0.999822 0.974925      fraud F1
Activity · grouped out-of-fold  0.809173 0.713952      macro F1


## 3 · Export and predict
The fraud forest is refitted on all labeled transactions with the development threshold fixed. At prediction time supply the earlier unlabeled history and a new batch using the original fields. Load identifiers as strings to preserve leading zeros. Each event should appear once across the two inputs.

For activity, the export averages decision scores from the four fitted fold models for new rows. Pass sensor columns in the same order as the headerless training file. The reported grouped score uses only the held-out fold model for each row. The four-model average has no independent test score.

In [9]:
class FraudPredictor:
    def __init__(self, forest, threshold, columns, fields):
        self.forest, self.threshold = forest, threshold
        self.columns, self.fields = columns, fields

    def predict(self, transactions, history):
        """Return 0/1 in batch order. Supply historical events once, excluding the new batch."""
        if len(transactions) == 0:
            return np.array([], dtype=int)
        events = pd.concat([history[self.fields], transactions[self.fields]], ignore_index=True)
        features = fraud_features(events).iloc[len(history):][self.columns].to_numpy(dtype="float32")
        return (self.forest.predict_proba(features)[:, 1] >= self.threshold).astype(int)

class ActivityPredictor:
    def __init__(self, models, labels):
        self.models, self.labels = models, labels

    def predict(self, sensors):
        """Return activity names for a 2-D numeric sensor matrix."""
        with threadpool_limits(limits=1, user_api="blas"):
            scores = [activity_scores(model, np.asarray(sensors)) for model in self.models]
        return self.labels[np.mean(scores, axis=0).argmax(axis=1)]

fraud_model = FraudPredictor(new_forest().fit(X, y), threshold, fraud_columns, event_fields)
activity_model = ActivityPredictor(fold_models, labels)
for filename, fitted in [("fraud_best_random_forest_transaction_history.pkl", fraud_model), ("activity_best_15_neighbor_sports_booster_ensemble.pkl", activity_model)]:
    with (OUTPUT_DIR / filename).open("wb") as stream:
        cloudpickle.dump(fitted, stream, protocol=5)

In [10]:
# Loading these files requires the packages above. Training cells are unnecessary for inference.
with (OUTPUT_DIR / "fraud_best_random_forest_transaction_history.pkl").open("rb") as stream:
    loaded_fraud = pickle.load(stream)
with (OUTPUT_DIR / "activity_best_15_neighbor_sports_booster_ensemble.pkl").open("rb") as stream:
    loaded_activity = pickle.load(stream)

# Small round-trip checks include the raw-data feature path and the tuned decision threshold.
batch = fraud.iloc[500:520]
history = fraud.iloc[:500]
expected = fraud_model.forest.predict_proba(X[[*range(500, 520)]])[:, 1] >= threshold
np.testing.assert_array_equal(loaded_fraud.predict(batch, history), expected)
readings = raw.iloc[:20]
np.testing.assert_array_equal(loaded_activity.predict(readings), activity_model.predict(readings))
print("Both pickle prediction checks passed.", OUTPUT_DIR.resolve())

Both pickle prediction checks passed. C:\Users\johnm\Documents\GitHub\PS5_TeamDragonaes-Co-Pilots\Code Prototypes\best_models
